# 4. Ensemble of BERT and DeBERTa

**NLP Final Term Project, Group 02.**

A weighted soft-vote over the two fine-tuned models:

$$P_{\text{ensemble}} = w \cdot P_{\text{BERT}} + (1 - w) \cdot P_{\text{DeBERTa}}$$

Soft rather than hard voting, because with only two members a majority vote has no
way to break a one-to-one tie.

Procedure, and the order matters:

1. take the two best configurations, each already selected on validation in notebooks
   02 and 03,
2. sweep `w` from 0 to 1 in steps of 0.05 and keep whichever maximises **validation**
   weighted F1,
3. apply that single fixed `w` to the test set exactly once,
4. compare the ensemble against its stronger member with a paired test, since both
   models predicted the same test rows.

The test set is used once, for reporting. It selects nothing.

**Requires** notebooks 02 and 03 to have run; this notebook trains nothing and reads
only their saved probability files.

## 4.1 Environment and paths

In [1]:
import os

os.environ.setdefault('HF_HOME', '/media/filwel/MLProject1/hf_cache')
os.environ.setdefault('HF_HUB_DISABLE_SYMLINKS', '1')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import gc
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# Raw corpora live outside the repository; the repository holds the derived splits.
PROJECT_DIR = Path('/media/filwel/All/Sakib/Semester 10/ NATURAL LANGUAGE PROCESSING /Project ')

FINAL_DIR = Path('/media/filwel/All/Sakib/Semester 10/ NATURAL LANGUAGE PROCESSING /Final')
if not FINAL_DIR.exists():
    p = Path.cwd().resolve()
    while p.name != 'Final' and p != p.parent:
        p = p.parent
    FINAL_DIR = p

PS_DIR = FINAL_DIR / 'experiments' / 'paper_scale'
WORK_DIR = PS_DIR / 'work'
RESULTS_DIR = PS_DIR / 'results'
PROBS_DIR = PS_DIR / 'probs'
MODELS_DIR = PS_DIR / 'models'
CKPT_DIR = Path('/media/filwel/MLProject1/nlp_paper_ckpt')

MAX_LEN = 128
EPOCHS = 5
WARMUP_RATIO = 0.1
PATIENCE = 2
SPLIT_SEED = 42
TRAIN_SEED = 42

MODELS = {'BERT': 'bert-base-uncased', 'DeBERTa': 'microsoft/deberta-v3-base'}
DATASET_NAMES = {'D1': 'DAIGT V2', 'D2': 'HC3'}

## 4.2 Load the two members' saved probabilities

Keys are rebuilt from the same configuration dictionaries that notebooks 02 and 03
used, so the ensemble is guaranteed to be built from the deployed checkpoints and not
from some other cell of the grid. The label vectors are asserted equal, which is the
check that the two members really were scored on the same rows.

In [2]:
BERT_BEST = {
    'D1': {'lr': 3e-5, 'bs': 32, 'wd': 0.1},
    'D2': {'lr': 2e-5, 'bs': 16, 'wd': 0.1},
}
DEBERTA_BEST = {
    'D1': {'lr': 3e-5, 'bs': 16, 'wd': 0.01},
    'D2': {'lr': 3e-5, 'bs': 16, 'wd': 0.1},
}
BEST = {'BERT': BERT_BEST, 'DeBERTa': DEBERTA_BEST}
MEMBERS = ('BERT', 'DeBERTa')


def run_key(tag, model_key, seed=TRAIN_SEED):
    cfg = BEST[model_key][tag]
    return (f'full_{tag}_{model_key}_lr{cfg["lr"]:g}_bs{cfg["bs"]}'
            f'_wd{cfg["wd"]:g}_s{seed}')


def load_probs(key):
    p = PROBS_DIR / f'{key}.npz'
    if not p.exists():
        raise FileNotFoundError(
            f'{p.name} not found. Run 02_bert_best_config.ipynb and '
            f'03_deberta_best_config.ipynb first.')
    z = np.load(p)
    return {k: z[k] for k in z.files}


PROBS, LABELS = {}, {}
for tag in ('D1', 'D2'):
    PROBS[tag] = {mk: load_probs(run_key(tag, mk)) for mk in MEMBERS}
    assert np.array_equal(PROBS[tag]['BERT']['val_labels'],
                          PROBS[tag]['DeBERTa']['val_labels']), 'validation labels differ'
    assert np.array_equal(PROBS[tag]['BERT']['test_labels'],
                          PROBS[tag]['DeBERTa']['test_labels']), 'test labels differ'
    LABELS[tag] = {'val': PROBS[tag]['BERT']['val_labels'],
                   'test': PROBS[tag]['BERT']['test_labels']}
    print(f'{tag} {DATASET_NAMES[tag]:9s} val={len(LABELS[tag]["val"]):5d}  '
          f'test={len(LABELS[tag]["test"]):5d}   '
          f'members: {run_key(tag, "BERT")} | {run_key(tag, "DeBERTa")}')

D1 DAIGT V2  val= 2800  test= 6998   members: full_D1_BERT_lr3e-05_bs32_wd0.1_s42 | full_D1_DeBERTa_lr3e-05_bs16_wd0.01_s42
D2 HC3       val= 4289  test=10732   members: full_D2_BERT_lr2e-05_bs16_wd0.1_s42 | full_D2_DeBERTa_lr3e-05_bs16_wd0.1_s42


## 4.3 Metrics and the paired test

McNemar's exact test asks only about the rows where the two systems disagree: of
those, is the split between them further from even than chance would allow. The
paired bootstrap resamples test rows 10,000 times and reports a 95 percent interval
on the difference in error rate. Both are paired, which is the right family of test
here because the same test rows went through both systems.

In [3]:
from scipy.stats import binomtest
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_recall_fscore_support)

N_BOOT = 10000


def weighted_metrics(y, p):
    acc = accuracy_score(y, p)
    pre, rec, f1, _ = precision_recall_fscore_support(
        y, p, average='weighted', zero_division=0)
    return acc, pre, rec, f1


def paired_test(y, pred_a, pred_b, n_boot=N_BOOT, seed=SPLIT_SEED):
    """McNemar exact plus a paired bootstrap on the error difference, a minus b."""
    rng = np.random.default_rng(seed)
    wa, wb = pred_a != y, pred_b != y
    b = int((~wa & wb).sum())      # a right, b wrong
    c = int((wa & ~wb).sum())      # a wrong, b right
    p = binomtest(b, b + c, 0.5).pvalue if (b + c) else 1.0
    idx = rng.integers(0, len(y), size=(n_boot, len(y)))
    boot = wa[idx].mean(1) - wb[idx].mean(1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {'a_right_b_wrong': b, 'a_wrong_b_right': c, 'mcnemar_exact_p': float(p),
            'error_diff_pp': float((wa.mean() - wb.mean()) * 100),
            'ci_lo_pp': float(lo * 100), 'ci_hi_pp': float(hi * 100),
            'ci_excludes_zero': bool(lo > 0 or hi < 0)}

## 4.4 Select the mixing weight on validation, then test once

In [4]:
WEIGHTS = np.round(np.arange(0.0, 1.0001, 0.05), 2)
ENSEMBLE, VAL_CURVE = {}, {}

for tag in ('D1', 'D2'):
    pb, pdb = PROBS[tag]['BERT'], PROBS[tag]['DeBERTa']
    yv, yt = LABELS[tag]['val'], LABELS[tag]['test']

    val_f1 = [weighted_metrics(yv, (w * pb['val_probs']
                                    + (1 - w) * pdb['val_probs']).argmax(1))[3]
              for w in WEIGHTS]
    best_w = float(WEIGHTS[int(np.argmax(val_f1))])
    VAL_CURVE[tag] = dict(zip(WEIGHTS.tolist(), [round(v, 4) for v in val_f1]))

    ens_pred = (best_w * pb['test_probs'] + (1 - best_w) * pdb['test_probs']).argmax(1)
    acc, pre, rec, f1 = weighted_metrics(yt, ens_pred)

    member_pred = {mk: PROBS[tag][mk]['test_probs'].argmax(1) for mk in MEMBERS}
    member_f1 = {mk: weighted_metrics(yt, v)[3] for mk, v in member_pred.items()}
    stronger = max(member_f1, key=member_f1.get)

    ENSEMBLE[tag] = {
        'weight_bert': best_w, 'weight_deberta': round(1 - best_w, 2),
        'degenerate': best_w in (0.0, 1.0),
        'test_accuracy': round(acc, 4), 'test_precision': round(pre, 4),
        'test_recall': round(rec, 4), 'test_f1': round(f1, 4),
        'member_f1': {mk: round(v, 4) for mk, v in member_f1.items()},
        'stronger_member': stronger,
        'ensemble_minus_stronger_f1': round(f1 - member_f1[stronger], 4),
        'confusion': confusion_matrix(yt, ens_pred).tolist(),
        'paired_vs_stronger': paired_test(yt, ens_pred, member_pred[stronger])}

    e = ENSEMBLE[tag]
    pt = e['paired_vs_stronger']
    print(f'{tag} {DATASET_NAMES[tag]}')
    print(f'   weight BERT={best_w:.2f}  weight DeBERTa={1 - best_w:.2f}'
          + ('   [weight sits at an endpoint]' if e['degenerate'] else ''))
    print(f'   BERT {member_f1["BERT"]:.4f}   DeBERTa {member_f1["DeBERTa"]:.4f}   '
          f'ensemble {f1:.4f}   (ensemble minus stronger member '
          f'{e["ensemble_minus_stronger_f1"]:+.4f})')
    print(f'   against {stronger}: McNemar p={pt["mcnemar_exact_p"]:.4g}  '
          f'error difference {pt["error_diff_pp"]:+.3f} pp  '
          f'95 percent CI [{pt["ci_lo_pp"]:+.3f}, {pt["ci_hi_pp"]:+.3f}]')
    print()

D1 DAIGT V2
   weight BERT=0.50  weight DeBERTa=0.50
   BERT 0.9916   DeBERTa 0.9917   ensemble 0.9936   (ensemble minus stronger member +0.0019)
   against DeBERTa: McNemar p=0.08543  error difference -0.186 pp  95 percent CI [-0.386, +0.014]



D2 HC3
   weight BERT=0.00  weight DeBERTa=1.00   [weight sits at an endpoint]
   BERT 0.9916   DeBERTa 0.9972   ensemble 0.9972   (ensemble minus stronger member +0.0000)
   against DeBERTa: McNemar p=1  error difference +0.000 pp  95 percent CI [+0.000, +0.000]



## 4.5 The validation weight sweep

Printed in full because the shape of this curve is the evidence for how the weight
was chosen, and on HC3 it is also the evidence that the flat region is wide.

In [5]:
curve = pd.DataFrame(VAL_CURVE)
curve.index.name = 'w_bert'
curve.columns = [f'{c} {DATASET_NAMES[c]} val_f1' for c in curve.columns]
print(curve.to_string())

        D1 DAIGT V2 val_f1  D2 HC3 val_f1
w_bert                                   
0.00                0.9943         0.9979
0.05                0.9943         0.9979
0.10                0.9943         0.9979
0.15                0.9943         0.9979
0.20                0.9946         0.9979
0.25                0.9950         0.9979
0.30                0.9946         0.9979
0.35                0.9946         0.9979
0.40                0.9946         0.9979
0.45                0.9950         0.9979
0.50                0.9961         0.9979
0.55                0.9961         0.9956
0.60                0.9961         0.9949
0.65                0.9957         0.9942
0.70                0.9957         0.9942
0.75                0.9957         0.9944
0.80                0.9957         0.9944
0.85                0.9957         0.9942
0.90                0.9957         0.9939
0.95                0.9957         0.9939
1.00                0.9957         0.9939


## 4.6 Ensemble results

In [6]:
rows = []
for tag in ('D1', 'D2'):
    e = ENSEMBLE[tag]
    rows.append({'dataset': f'{tag} {DATASET_NAMES[tag]}',
                 'w_bert': e['weight_bert'], 'w_deberta': e['weight_deberta'],
                 'BERT_f1': e['member_f1']['BERT'],
                 'DeBERTa_f1': e['member_f1']['DeBERTa'],
                 'ensemble_f1': e['test_f1'],
                 'ensemble_accuracy': e['test_accuracy'],
                 'vs_stronger': e['ensemble_minus_stronger_f1'],
                 'mcnemar_p': round(e['paired_vs_stronger']['mcnemar_exact_p'], 4)})

print(pd.DataFrame(rows).to_string(index=False))
print()
for tag in ('D1', 'D2'):
    cm = np.array(ENSEMBLE[tag]['confusion'])
    print(f'{tag} {DATASET_NAMES[tag]} ensemble test confusion')
    print(pd.DataFrame(cm, index=['true_human', 'true_ai'],
                       columns=['pred_human', 'pred_ai']).to_string())
    print()

    dataset  w_bert  w_deberta  BERT_f1  DeBERTa_f1  ensemble_f1  ensemble_accuracy  vs_stronger  mcnemar_p
D1 DAIGT V2     0.5        0.5   0.9916      0.9917       0.9936             0.9936       0.0019     0.0854
     D2 HC3     0.0        1.0   0.9916      0.9972       0.9972             0.9972       0.0000     1.0000

D1 DAIGT V2 ensemble test confusion
            pred_human  pred_ai
true_human        3487       32
true_ai             13     3466

D2 HC3 ensemble test confusion
            pred_human  pred_ai
true_human        5360       17
true_ai             13     5342



## 4.7 Reading the result

An ensemble helps only when two conditions hold together: the members are comparable
in strength, and they make different mistakes. The two datasets sit on opposite sides
of the first condition, which is why they behave differently here.

**DAIGT V2.** The members are almost exactly matched, 0.9916 against 0.9917, and they
do make different mistakes, so the validation sweep settles on an even split and the
mixture improves on both of them: 0.9936, which is 0.19 percentage points above the
stronger member. The two systems disagree on 49 test rows and the ensemble wins 31 of
them. That is a favourable direction, but McNemar's exact test returns p = 0.085 and
the bootstrap interval on the error difference runs from -0.39 to +0.01 percentage
points, so it spans zero. The honest statement is that the ensemble is not worse and
is probably slightly better, not that the gain is established on this test set.

**HC3.** DeBERTa is ahead by 0.56 percentage points, which is a wide margin at this
error rate, and letting BERT vote can only pull predictions toward the weaker model.
Validation F1 is flat at 0.9979 for every weight from 0 up to 0.50 and falls after
that, so the argmax lands on the first of the tied weights, 0. The resulting test
predictions are identical to DeBERTa's, the two disagree on zero rows, and McNemar
has nothing left to test.

A weight at an endpoint is not a failed run and should not be described as the
ensemble collapsing. It is the selection rule working: given a member this dominant,
the best available mixture leans entirely on it, and the flat validation band exists
precisely because so few predictions change across most of the range. What would
improve on this is more diversity between members, not a different mixing rule -- a
third family with a different inductive bias, or the same architecture trained on
different folds of the data.

`docs/ENSEMBLE_EXPLAINED.md` carries the longer version of this argument. It was
written against the earlier small-scale sweep, where DAIGT selected a 0.35 / 0.65 mix
and the ensemble sat just below DeBERTa; its reasoning still holds, but the DAIGT
numbers there are superseded by the full-scale ones reported above.

In [7]:
print(json.dumps(ENSEMBLE, indent=2, default=float))

{
  "D1": {
    "weight_bert": 0.5,
    "weight_deberta": 0.5,
    "degenerate": false,
    "test_accuracy": 0.9936,
    "test_precision": 0.9936,
    "test_recall": 0.9936,
    "test_f1": 0.9936,
    "member_f1": {
      "BERT": 0.9916,
      "DeBERTa": 0.9917
    },
    "stronger_member": "DeBERTa",
    "ensemble_minus_stronger_f1": 0.0019,
    "confusion": [
      [
        3487,
        32
      ],
      [
        13,
        3466
      ]
    ],
    "paired_vs_stronger": {
      "a_right_b_wrong": 31,
      "a_wrong_b_right": 18,
      "mcnemar_exact_p": 0.08543313315739454,
      "error_diff_pp": -0.18576736210345812,
      "ci_lo_pp": -0.38582452129179773,
      "ci_hi_pp": 0.0142897970848814,
      "ci_excludes_zero": false
    }
  },
  "D2": {
    "weight_bert": 0.0,
    "weight_deberta": 1.0,
    "degenerate": true,
    "test_accuracy": 0.9972,
    "test_precision": 0.9972,
    "test_recall": 0.9972,
    "test_f1": 0.9972,
    "member_f1": {
      "BERT": 0.9916,
      "DeBERT